# Neural Network (LightGCN) recommender

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict
import random
from torch_geometric.nn import LightGCN
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

## Open and parse data for better mapping

In [2]:
ratings = pd.read_csv(
    "data/ratings.dat",
    sep="::",
    engine="python",
    names=["user_id", "movie_id", "rating", "timestamp"]
)
ratings["timestamp"] = pd.to_datetime(ratings["timestamp"], unit="s")
print(ratings)


         user_id  movie_id  rating           timestamp
0              1      1193       5 2000-12-31 22:12:40
1              1       661       3 2000-12-31 22:35:09
2              1       914       3 2000-12-31 22:32:48
3              1      3408       4 2000-12-31 22:04:35
4              1      2355       5 2001-01-06 23:38:11
...          ...       ...     ...                 ...
1000204     6040      1091       1 2000-04-26 02:35:41
1000205     6040      1094       5 2000-04-25 23:21:27
1000206     6040       562       5 2000-04-25 23:19:06
1000207     6040      1096       4 2000-04-26 02:20:48
1000208     6040      1097       4 2000-04-26 02:19:29

[1000209 rows x 4 columns]


In [3]:
ratings = ratings[ratings["rating"] >= 4]
print(ratings)

         user_id  movie_id  rating           timestamp
0              1      1193       5 2000-12-31 22:12:40
3              1      3408       4 2000-12-31 22:04:35
4              1      2355       5 2001-01-06 23:38:11
6              1      1287       5 2000-12-31 22:33:59
7              1      2804       5 2000-12-31 22:11:59
...          ...       ...     ...                 ...
1000202     6040      1089       4 2000-04-25 23:23:16
1000205     6040      1094       5 2000-04-25 23:21:27
1000206     6040       562       5 2000-04-25 23:19:06
1000207     6040      1096       4 2000-04-26 02:20:48
1000208     6040      1097       4 2000-04-26 02:19:29

[575281 rows x 4 columns]


## Convert users and movie_ids into numeric indices onto a combined graph space

In [4]:
movies = ratings['movie_id'].unique()
users = ratings['user_id'].unique()

m_idx = {}
for index, movie in enumerate(movies):
    m_idx[movie] = index
    
u_idx = {}
for index, user in enumerate(users):
    u_idx[user] = index

ratings['m_idx'] = ratings['movie_id'].map(m_idx)
ratings['u_idx'] = ratings['user_id'].map(u_idx)
ratings['movie_node'] = ratings['m_idx'] + len(users)

## Create a bidirectional graph edge list connecting users and movies

In [5]:
dest = torch.tensor(ratings['movie_node'].values, dtype=torch.long)
source = torch.tensor(ratings['u_idx'].values, dtype=torch.long)
edge = torch.stack([torch.cat([source, dest]), torch.cat([dest, source])], dim=0)
num_nodes = len(movies) + len(users)

## Group user interactions, build a training and testing set 

In [6]:
u_i = defaultdict(list)
for i in ratings.itertuples():
    u_i[i.u_idx].append(i.m_idx)

test_p = []
train_p = []
random.seed(42)

for u, i in u_i.items():
    items = list(set(i))
    if len(items) < 2:
        continue
    t_i = random.choice(items)
    test_p.append((u, t_i))
    for i in items:
        if i != t_i:
            train_p.append((u, i))
df_test = pd.DataFrame(test_p, columns=['u_idx', 'm_idx'])
df_train = pd.DataFrame(train_p, columns=['u_idx', 'm_idx'])

In [7]:
df_train['movie_node'] = df_train['m_idx'] + len(users)
dest_train = torch.tensor(df_train['movie_node'].values, dtype=torch.long)
source_train = torch.tensor(df_train['u_idx'].values, dtype=torch.long)

## Create model

In [8]:
e_idx = torch.stack([torch.cat([source_train, dest_train]), torch.cat([dest_train, source_train])], dim=0)
device = torch.device('cpu')
gcn = LightGCN(num_nodes=len(movies)+len(users), embedding_dim=64, num_layers=3).to(device)
e_idx = e_idx.to(device)


## Negative sampling

In [9]:
def n_items(users_batch, train_dict, num_movies=len(movies)):
    neg_items = []
    
    for u in users_batch.tolist():
        while True:
            neg = random.randint(0, num_movies - 1)
            if neg not in train_dict[u]:
                neg_items.append(neg)
                break
                
    return torch.tensor(neg_items, dtype=torch.long)

## Convert training user-item pairs into tensor form and build graph edges

In [10]:
p_item = torch.tensor(
        df_train['m_idx'].values + len(users),
        dtype=torch.long
)
p_user = torch.tensor(
        df_train['u_idx'].values,
        dtype=torch.long
)
p_e_idx = torch.stack([p_user, p_item], dim=0).to(device)

## Initialize Adam optimizer and parameters

In [11]:
optimizer = optim.Adam(gcn.parameters(), lr=0.001)
train = defaultdict(set)
for i in df_train.itertuples():
    train[i.u_idx].add(i.m_idx)
    
range_m_idx = set(range(len(movies)))
batch_size = 4096

## Train the model on 10 epochs

In [12]:
for epoch in range(10):
    gcn.train()
    total_loss = 0

    perm = torch.randperm(p_e_idx.size(1))

    for start in range(0, p_e_idx.size(1), batch_size):
        batch_idx = perm[start:start + batch_size]
        batch = p_e_idx[:, batch_idx]

        users_batch = batch[0]                     
        pos_items = batch[1] - len(users)           

        neg_items = n_items(users_batch, train)     

        optimizer.zero_grad()
        out = gcn.get_embedding(edge_index=p_e_idx)

        user_embeds = out[:len(users)]
        movie_embeds = out[len(users):]

        user_emb = user_embeds[users_batch]
        pos_emb = movie_embeds[pos_items]
        neg_emb = movie_embeds[neg_items]

        pos_scores = (user_emb * pos_emb).sum(dim=1)
        neg_scores = (user_emb * neg_emb).sum(dim=1)

        loss = -torch.log(torch.sigmoid(pos_scores - neg_scores) + 1e-8).mean()

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 96.1985
Epoch 2, Loss: 89.6824
Epoch 3, Loss: 72.9490
Epoch 4, Loss: 59.3171
Epoch 5, Loss: 51.6036
Epoch 6, Loss: 47.5699
Epoch 7, Loss: 45.2968
Epoch 8, Loss: 43.9391
Epoch 9, Loss: 42.9694
Epoch 10, Loss: 42.3057


## Build a user's candidate set and create a function to generate top reccomendations

In [13]:
u_c ={}
num_users = len(users)
for i in df_test['u_idx'].unique():
    watched = train.get(i, set())
    u_c[i] = torch.tensor(list(range_m_idx - watched), dtype=torch.long) + num_users

In [14]:
train_watched = defaultdict(set)
for i in df_train.itertuples():
    train_watched[i.u_idx].add(i.m_idx)

u_c = {}
num_users = len(users)
range_m_idx = set(range(len(movies)))

for i in df_test['u_idx'].unique():

    watched = train_watched.get(i, set())
    u_c[i] = list(range_m_idx - watched)

def gcn_rec(u_idx, k=10):
    cand = u_c.get(u_idx, [])
    if len(cand) == 0:
        return []
    
    gcn.eval()
    shift_m = [i + num_users for i in cand]
    nodes_u = torch.tensor([u_idx], dtype=torch.long, device=device)
    nodes_c = torch.tensor(shift_m, dtype=torch.long, device=device)
    with torch.no_grad():
        rec = gcn.recommend(edge_index=e_idx, src_index=nodes_u, dst_index=nodes_c, k=min(k, len(cand)))
    top = (rec.view(-1) - len(users)).cpu()
    return top.tolist()
    


In [15]:
def ndcg_at_k(recommended, relevant, k=10):
    recommended = recommended[:k]
    relevant = set(relevant)
    dcg = 0
    for i, item in enumerate(recommended):
        if item in relevant:
            dcg += 1 / np.log2(i + 2)

    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / np.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0


## Test the model of top 10 and top 100 recommendations for each user in test set

In [17]:
gcn.eval()
with torch.no_grad():
    out = gcn.get_embedding(edge_index=e_idx)
    user_embeds = out[:len(users)]
    movie_embeds = out[len(users):]

total_hits = 0
total_relevant = 0
precision_sum = 0
recall_sum = 0
ndcg_sum = 0
n_users = 0
test = df_test.groupby('u_idx')['m_idx'].apply(list).to_dict()
for u_idx, true_m_indices in test.items():
    if u_idx not in u_c:
        continue

    u_vec = user_embeds[u_idx]
    cand_m_indices = u_c[u_idx]

    if len(cand_m_indices) == 0:
        continue

    cand_embeds = movie_embeds[cand_m_indices]
    scores = torch.matmul(cand_embeds, u_vec)

    top_k = min(10, len(scores))
    top_indices = torch.topk(scores, k=top_k).indices
    recommendations = [cand_m_indices[i] for i in top_indices.tolist()]

    if isinstance(true_m_indices, (list, set, tuple)):
        relevant_movies = set(true_m_indices)
    else:
        relevant_movies = {true_m_indices}
    hits = len(relevant_movies.intersection(recommendations))

    precision = hits / 10
    recall = hits / len(relevant_movies) if len(relevant_movies) > 0 else 0
    ndcg = ndcg_at_k(recommendations, relevant_movies, k=10)

    total_hits += hits
    total_relevant += len(relevant_movies)
    precision_sum += precision
    recall_sum += recall
    ndcg_sum += ndcg
    n_users += 1

avg_precision = precision_sum / n_users if n_users > 0 else 0
avg_recall = recall_sum / n_users if n_users > 0 else 0
avg_ndcg = ndcg_sum / n_users if n_users > 0 else 0
micro_recall = total_hits / total_relevant if total_relevant > 0 else 0

print("Users evaluated:", n_users)
print("Hits:", total_hits)
print("Possible hits:", total_relevant)
print(f"Precision@10: {avg_precision:.4f}")
print(f"Recall@10: {avg_recall:.4f}")
print(f"NDCG@10: {avg_ndcg:.4f}")
print(f"Micro Recall@10: {micro_recall:.4f}")

Users evaluated: 6037
Hits: 509
Possible hits: 6037
Precision@10: 0.0084
Recall@10: 0.0843
NDCG@10: 0.0442
Micro Recall@10: 0.0843


In [18]:
gcn.eval()
with torch.no_grad():
    out = gcn.get_embedding(edge_index=e_idx)
    user_embeds = out[:len(users)]
    movie_embeds = out[len(users):]

total_hits = 0
total_relevant = 0
precision_sum = 0
recall_sum = 0
ndcg_sum = 0
n_users = 0
test = df_test.groupby('u_idx')['m_idx'].apply(list).to_dict()
for u_idx, true_m_indices in test.items():
    if u_idx not in u_c:
        continue

    u_vec = user_embeds[u_idx]
    cand_m_indices = u_c[u_idx]

    if len(cand_m_indices) == 0:
        continue

    cand_embeds = movie_embeds[cand_m_indices]
    scores = torch.matmul(cand_embeds, u_vec)

    top_k = min(100, len(scores))
    top_indices = torch.topk(scores, k=top_k).indices
    recommendations = [cand_m_indices[i] for i in top_indices.tolist()]

    if isinstance(true_m_indices, (list, set, tuple)):
        relevant_movies = set(true_m_indices)
    else:
        relevant_movies = {true_m_indices}
    hits = len(relevant_movies.intersection(recommendations))

    precision = hits / 100
    recall = hits / len(relevant_movies) if len(relevant_movies) > 0 else 0
    ndcg = ndcg_at_k(recommendations, relevant_movies, k=100)

    total_hits += hits
    total_relevant += len(relevant_movies)
    precision_sum += precision
    recall_sum += recall
    ndcg_sum += ndcg
    n_users += 1

avg_precision = precision_sum / n_users if n_users > 0 else 0
avg_recall = recall_sum / n_users if n_users > 0 else 0
avg_ndcg = ndcg_sum / n_users if n_users > 0 else 0
micro_recall = total_hits / total_relevant if total_relevant > 0 else 0

print("Users evaluated:", n_users)
print("Hits:", total_hits)
print("Possible hits:", total_relevant)
print(f"Precision@100: {avg_precision:.4f}")
print(f"Recall@100: {avg_recall:.4f}")
print(f"NDCG@100: {avg_ndcg:.4f}")
print(f"Micro Recall@100: {micro_recall:.4f}")

Users evaluated: 6037
Hits: 2111
Possible hits: 6037
Precision@100: 0.0035
Recall@100: 0.3497
NDCG@100: 0.0955
Micro Recall@100: 0.3497
